# Predicting long term movie performance using as few early reviews

## Data Cleaning

In [1]:
from pathlib import Path
import re
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split

### Cleaning Movie and Review Data

- Load Horror, Animation, and Adventure movie tables and label each row with its source genre
- Resolve cross-genre duplicates so each movie keeps one preferred genre
- Parse runtime into minutes; clean MPAA rating, release year, and IMDb rating
- Assign an id for each movie
- Load matching review CSVs, drop reviews without a star rating or date
- Clean HTML from review text, order reviews by date within each movie
- Label movies positive if star rating ≥ 7

In [2]:
# load one CSV per genre and tag rows with that genre
movie_frames = []
for genre in ["Horror", "Animation", "Adventure"]:
    df = pd.read_csv(Path("movies_per_genre") / f"{genre}.csv")
    df["genre"] = genre
    movie_frames.append(df)

movies = pd.concat(movie_frames, ignore_index=True)

# prefer the first matching genre that appears if the title appears in more than one genre
def preferred_genre(row):
    tags = [t.strip() for t in str(row["genres"]).split(";") if t.strip()]
    for tag in tags:
        if tag in ["Horror", "Animation", "Adventure"]:
            return tag
    return row["genre"]

movies["genre"] = movies.apply(preferred_genre, axis=1)
movies = movies.drop_duplicates(subset=["name", "year"], keep="first").reset_index(drop=True)
movies.head()

,name,year,movie_rated,run_length,genres,release_date,rating,num_raters,num_reviews,review_url,genre
0,The Shining,1980,R,2h 26min,Drama; Horror;,13 June 1980 (USA),8.4,860290,1805,https://www.imdb.com/title/tt0081505/reviews/_...,Horror
1,Alien,1979,R,1h 57min,Horror; Sci-Fi;,22 June 1979 (USA),8.4,762793,1423,https://www.imdb.com/title/tt0078748/reviews/_...,Horror
2,World War Z,2013,PG-13,1h 56min,Action; Adventure; Horror;,21 June 2013 (USA),7.0,586801,1125,https://www.imdb.com/title/tt0816711/reviews/_...,Adventure
3,Psycho,1960,R,1h 49min,Horror; Mystery; Thriller;,8 September 1960 (USA),8.5,580951,1263,https://www.imdb.com/title/tt0054215/reviews/_...,Horror
4,Shaun of the Dead,2004,R,1h 39min,Comedy; Horror;,24 September 2004 (USA),7.9,499244,971,https://www.imdb.com/title/tt0365748/reviews/_...,Horror


In [3]:
# convert strings like into total minutes
def parse_runtime_minutes(value):
    if pd.isna(value):
        return np.nan
    text = str(value).strip().lower()
    hours = re.search(r"(\d+)\s*h", text)
    minutes = re.search(r"(\d+)\s*min", text)
    total = 0
    if hours:
        total += int(hours.group(1)) * 60
    if minutes:
        total += int(minutes.group(1))
    return total if total > 0 else np.nan


# clean and rename metadata columns
movies["movie_id"] = np.arange(len(movies), dtype=int)
movies["runtime"] = movies["run_length"].map(parse_runtime_minutes)
movies["mpaa_rating"] = movies["movie_rated"].astype(str).str.strip()
movies["release_year"] = movies["year"].astype(int)
movies["imdb_rating"] = movies["rating"].astype(float)
movies["num_raters"] = movies["num_raters"].astype(int)
movies["num_reviews_listed"] = movies["num_reviews"].astype(int)

movies.head()

,name,year,movie_rated,run_length,genres,release_date,rating,num_raters,num_reviews,review_url,genre,movie_id,runtime,mpaa_rating,release_year,imdb_rating,num_reviews_listed
0,The Shining,1980,R,2h 26min,Drama; Horror;,13 June 1980 (USA),8.4,860290,1805,https://www.imdb.com/title/tt0081505/reviews/_...,Horror,0,146,R,1980,8.4,1805
1,Alien,1979,R,1h 57min,Horror; Sci-Fi;,22 June 1979 (USA),8.4,762793,1423,https://www.imdb.com/title/tt0078748/reviews/_...,Horror,1,117,R,1979,8.4,1423
2,World War Z,2013,PG-13,1h 56min,Action; Adventure; Horror;,21 June 2013 (USA),7.0,586801,1125,https://www.imdb.com/title/tt0816711/reviews/_...,Adventure,2,116,PG-13,2013,7.0,1125
3,Psycho,1960,R,1h 49min,Horror; Mystery; Thriller;,8 September 1960 (USA),8.5,580951,1263,https://www.imdb.com/title/tt0054215/reviews/_...,Horror,3,109,R,1960,8.5,1263
4,Shaun of the Dead,2004,R,1h 39min,Comedy; Horror;,24 September 2004 (USA),7.9,499244,971,https://www.imdb.com/title/tt0365748/reviews/_...,Horror,4,99,R,2004,7.9,971


In [4]:
# review filenames replace ':' with '_' in the movie title
# so we change it so that we can find the matching csv file
def review_filename(name, year):
    return f"{str(name).replace(':', '_')} {int(year)}.csv"


# clean review text: strip HTML tags and collapse whitespace
def clean_review_text(text):
    if pd.isna(text):
        return ""
    cleaned = re.sub(r"<br\s*/?>", " ", str(text), flags=re.IGNORECASE)
    cleaned = re.sub(r"<[^>]+>", " ", cleaned)
    cleaned = re.sub(r"\s+", " ", cleaned).strip()
    return cleaned


# load each movie's review file and attach movie_id / genre
review_frames = []
for row in movies.itertuples(index=False):
    path = Path("all_reviews_per_movie_raw") / review_filename(row.name, row.release_year)
    if not path.exists():
        continue
    rev = pd.read_csv(path)
    rev["movie_id"] = row.movie_id
    rev["genre"] = row.genre
    review_frames.append(rev)

reviews = pd.concat(review_frames, ignore_index=True)

# drop reviews with missing star rating or date
reviews["star_rating"] = pd.to_numeric(reviews["rating"], errors="coerce")
reviews = reviews.dropna(subset=["star_rating"]).copy()
reviews["star_rating"] = reviews["star_rating"].astype(int)

reviews["date"] = pd.to_datetime(reviews["date"], errors="coerce")
reviews = reviews.dropna(subset=["date"]).copy()

# clean text and create binary positive label
reviews["text"] = reviews["review"].map(clean_review_text)
reviews = reviews[reviews["text"].str.len() > 0].copy()
reviews["positive"] = (reviews["star_rating"] >= 7).astype(int)

# chronological order within each movie (1 = first review)
reviews = reviews.sort_values(["movie_id", "date", "username"], kind="mergesort")
reviews["review_order"] = reviews.groupby("movie_id").cumcount() + 1

reviews = reviews[
    ["movie_id", "genre", "review_order", "date", "star_rating", "positive", "text"]
].reset_index(drop=True)

# keep only movies that still have at least one usable review
movies = movies[movies["movie_id"].isin(reviews["movie_id"].unique())].reset_index(drop=True)
reviews.head()

,movie_id,genre,review_order,date,star_rating,positive,text
0,0,Horror,1,1998-08-04,10,1,"First of all, I loved the book. It remains my ..."
1,0,Horror,2,1998-08-07,10,1,Although it is sometimes considered to be a mo...
2,0,Horror,3,1998-09-07,10,1,"Stanley Kubrick's ""The Shining"" was a high-wat..."
3,0,Horror,4,1998-10-11,10,1,This is a remarkable milestone in cinema. Kubr...
4,0,Horror,5,1998-10-14,10,1,What can I say to all the fans of the novel Th...


### Train/Validation/Test Split

- Split by movie 
- Stratify on genre: 70% train, then split the remaining 30% into 15% validation / 15% test
- Split train movies in half into `lstm_train` and `mlp_train`
- Add a `split` column on movies and merge it onto reviews by `movie_id`

In [5]:
# 70/30 split stratified by genre
train_movies, temp_movies = train_test_split(
    movies, test_size=0.30, stratify=movies["genre"], random_state=42)

# split 30% in half (15% validation, 15% test)
val_movies, test_movies = train_test_split(
    temp_movies, test_size=0.50, stratify=temp_movies["genre"], random_state=42)

# split train in half: one half for LSTM and the other half for MLP
lstm_train_movies, mlp_train_movies = train_test_split(
    train_movies, test_size=0.50, stratify=train_movies["genre"], random_state=42)

# create column identifying which split each movie belongs to
movies = movies.copy()
movies["split"] = None
for subset, label in [
    (lstm_train_movies, "lstm_train"),
    (mlp_train_movies, "mlp_train"),
    (val_movies, "validation"),
    (test_movies, "test"),
]:
    movies.loc[movies["movie_id"].isin(subset["movie_id"]), "split"] = label

# copy each movie's split label to every review that has the same movie_id
reviews = reviews.merge(movies[["movie_id", "split"]], on="movie_id", how="left")

In [6]:
# to see the number of movies and reviews in each split by genre 
# movies.groupby(["split", "genre"]).size().unstack(fill_value=0)
# reviews.groupby(["split", "genre"]).size().unstack(fill_value=0)

### Movie Metadata DataFrame

- Set different values of k (number of earliest reviews)
- For each k value, keep movies with at least k reviews
- Build one row per (movie, k) pair with split, numeric metadata, and IMDb rating as the target
- One-hot encode genre and MPAA rating so MLP features are numeric
- Save `movies_clean.csv` and `reviews_clean.csv`

In [7]:
k_values = [1, 2, 3, 5, 7, 10, 15, 20, 30, 50]

# create a column with the number of reviews per movie
n_reviews = reviews.groupby("movie_id").size().rename("n_reviews")
# merge the number of reviews with the movies dataframe
movies_with_counts = movies.merge(n_reviews, on="movie_id", how="left")

meta_cols = [
    "movie_id", 
    "split", 
    "genre", 
    "runtime", 
    "mpaa_rating",
    "release_year", 
    "imdb_rating", 
    "num_raters", 
    "num_reviews_listed",]

# one row per (movie, k) for every k the movie has enough reviews for
mlp_rows = []
for k in k_values:
    eligible = movies_with_counts.loc[movies_with_counts["n_reviews"] >= k, meta_cols].copy()
    eligible["k"] = k
    mlp_rows.append(eligible)

mlp_base = pd.concat(mlp_rows, ignore_index=True)

# one-hot encode categoricals 
# leave split as a string for filtering
mlp_base = pd.get_dummies(mlp_base, columns=["genre", "mpaa_rating"], dtype=int)

mlp_base = mlp_base.sort_values(["movie_id", "k"]).reset_index(drop=True)

In [8]:
# save copy of clean movie and review data for model use
clean_dir = Path("data/clean")
clean_dir.mkdir(parents=True, exist_ok=True)

movies.to_csv(clean_dir / "movies_clean.csv", index=False)
reviews[
    ["movie_id", "genre", "split", "review_order", "date", "star_rating", "positive", "text"]
].to_csv(clean_dir / "reviews_clean.csv", index=False)

## What do the early star ratings alone tell us, and how sure can we be? (Bayesian)

## Does review data add anything? (LSTM)

## Final predictions! (MLP)